In [1]:
from rdkit import Chem
from rdkit.Chem import FilterCatalog, Descriptors, Crippen, rdMolDescriptors, BRICS
import json
from tqdm import tqdm
import random
from collections import namedtuple
import numpy as np
from pathlib import Path
import pickle
import sys

In [2]:
from sltools import property_tools
from sltools.property_tools import OUT_OF_RANGE, UNDESIRABLE_PATTERNS, CORRECT_PYRROLE, COVALENT_WARHEADS, params, PAINS_catalog
#from inference_tools import InferenceObject
COVALENT_WARHEADS

{'sulfonyl fluorides': '[#16](=[#8])(=[#8])-[#9]',
 'chloroacetamides': '[#8]=[#6](-[#6]-[#17])-[#7]',
 'cyanoacrylamides': '[#7]-[#6](=[#8])-[#6](-[#6]#[#7])=[#6]',
 'epoxides': '[#6]1-[#6]-[#8]-1',
 'aziridines': '[#6]1-[#6]-[#7]-1',
 'disulfides': '[#16]-[#16]',
 'aldehydes': '[#6](=[#8])-[#1]',
 'vinyl sulfones': '[#6]=[#6]-[#16](=[#8])(=[#8])-[#7]',
 'boronic acids/esters': '[#6]-[#5](-[#8])-[#8]',
 'acrylamides': '[#6]=[#6]-[#6](=[#8])-[#7]',
 'cyanamides': '[#6]-[#7](-[#6]#[#7])-[#6]',
 'chloroFluoroAcetamides': '[#7]-[#6](=[#8])-[#6](-[#9])-[#17]',
 'butynamides': '[#6]#[#6]-[#6](=[#8])-[#7]-[#6]',
 'chloropropionamides': '[#7]-[#6](=[#8])-[#6](-[#6])-[#17]',
 'fluorosulfates': '[#8]=[#16](=[#8])(-[#9])-[#8]',
 'beta lactams': '[#7]1-[#6]-[#6]-[#6]-1=[#8]'}

In [3]:
sft = []

#temp_strs = [str(round(0.1*t,1)) for t in range(5,10)]
temp_strs = ['1.0']
for temp_str in temp_strs:
    with open('instr_sft_data/temperature_'+temp_str+'.pkl', 'rb') as file:
        sft.append(pickle.load(file))

In [4]:
template = {
    'source':'',
    'instruction': '',
    'chosen_response':'',
    'rejected_response': '',
    'chosen_avg_rating': '',
    'rejected_avg_rating': '',
    'chosen_model': ''
    }
instruction = 'You love and excel at generating SMILES strings of drug-like molecules'
all_jsondata = []
for s in tqdm(sft):
    jsondata = []
    for m in s:
        winners = []
        losers = []
        for element in m[1]:
            if element[1] == False: losers.append(element)
            elif element[1] == True: winners.append(element)
            else:
                raise ValueError('expected either True or False for whether the smiles string satisfied the constraints')
        winners = list(set(winners))
        losers = list(set(losers))
        pairings = zip(winners, losers)
        pipestr = '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'+instruction+'<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'+m[0]+'<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
        jsondata += [{**template, 'instruction':pipestr,'chosen_response':nl[0][0], 'rejected_response':nl[1][0],
                     'chosen_avg_rating':int(nl[0][1]), 'rejected_avg_rating':int(nl[1][1])} for nl in pairings]
    all_jsondata.append(jsondata)
    print(len(jsondata))

100%|██████████| 1/1 [00:00<00:00,  5.31it/s]

116914


In [5]:
#generate the DPO dataset
with open('prompt_following/instr-t1.jsonl', 'w+') as out:
        for item in all_jsondata[0]:
            out.write(json.dumps(item) + '\n')